In [0]:
#End to end pipeline

In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("pyspark optimization") \
    .getOrCreate()

spark

In [0]:
from pyspark.sql.functions import rand

In [0]:

orders = spark.range(0, 5_000_000).withColumnRenamed("id", "order_id") \
    .withColumn("user_id", (rand()*100000).cast("int")) \
    .withColumn("product_id", (rand()*1000).cast("int")) \
    .withColumn("category", (rand()*10).cast("int")) \
    .withColumn("amount", rand()*1000)

In [0]:
# write orders
orders.write.mode("overwrite").parquet("/Volumes/workspace/default/pipeline/orders")

In [0]:
users = spark.range(0, 100000).withColumnRenamed("id", "user_id") \
    .withColumn("country", (rand()*5).cast("int"))

In [0]:
users.write.mode("overwrite").parquet("/Volumes/workspace/default/pipeline/users")

In [0]:
products = spark.range(0, 1000).withColumnRenamed("id", "product_id") \
    .withColumn("price", rand()*500)

In [0]:
products.write.mode("overwrite").parquet("/Volumes/workspace/default/pipeline/products")

In [0]:
orders = spark.read.parquet("/Volumes/workspace/default/pipeline/orders")
users = spark.read.parquet("/Volumes/workspace/default/pipeline/users")
products = spark.read.parquet("/Volumes/workspace/default/pipeline/products")

In [0]:
orders.show(6)

+--------+-------+----------+--------+------------------+
|order_id|user_id|product_id|category|            amount|
+--------+-------+----------+--------+------------------+
| 1875000|  69198|       706|       7|20.136769426318146|
| 1875001|  89418|       430|       8| 830.2348467817321|
| 1875002|  25525|       516|       3| 206.5022315265721|
| 1875003|  61122|       988|       9| 368.8139589934547|
| 1875004|  49922|       803|       2|196.63780016278542|
| 1875005|  33930|       855|       4| 36.14995627036188|
+--------+-------+----------+--------+------------------+
only showing top 6 rows


In [0]:
users.printSchema()

root
 |-- user_id: long (nullable = true)
 |-- country: integer (nullable = true)



In [0]:
from pyspark.sql.functions import col,lower,trim,broadcast,sum as _sum

In [0]:
# Transformation

# Step 1 - cleaning the category column

orders_clean1 = orders.withColumn("category", col("category").cast("string"))

In [0]:
orders_clean = orders_clean1.withColumn("category", lower(trim(col("category"))))
orders_clean.show(6)

+--------+-------+----------+--------+------------------+
|order_id|user_id|product_id|category|            amount|
+--------+-------+----------+--------+------------------+
| 1875000|  69198|       706|       7|20.136769426318146|
| 1875001|  89418|       430|       8| 830.2348467817321|
| 1875002|  25525|       516|       3| 206.5022315265721|
| 1875003|  61122|       988|       9| 368.8139589934547|
| 1875004|  49922|       803|       2|196.63780016278542|
| 1875005|  33930|       855|       4| 36.14995627036188|
+--------+-------+----------+--------+------------------+
only showing top 6 rows


In [0]:
# Repartition - it is wide transformation


orders_clean = orders_clean.repartition("user_id") #it will redistribute the data across the cluster based on user_id so that processing is more balanced


In [0]:
# optimizing joins

df = orders_clean.join(broadcast(users), "user_id") \
    .join(products, "product_id")

In [0]:
#caching

#df.cache()

In [0]:
#find top products by revenue
top_products= df.groupBy("product_id") \
    .count()\
    .orderBy(col("count").desc())\
    .limit(10)

#this is top n aggregration
#count is wide transformation

In [0]:
top_products.show(10)

+----------+-----+
|product_id|count|
+----------+-----+
|       238| 5227|
|       264| 5208|
|       948| 5197|
|       147| 5195|
|       858| 5176|
|       224| 5169|
|       160| 5167|
|       618| 5167|
|       868| 5162|
|        11| 5162|
+----------+-----+



In [0]:
#revenue per country
revenue_df = df.groupBy("country")\
    .agg(_sum("amount").alias("total_revenue"))

In [0]:
#write data
top_products.write.mode("overwrite").parquet("/Volumes/workspace/default/pipeline/output/top_products")
revenue_df.write.mode("overwrite").parquet("/Volumes/workspace/default/pipeline/output/revenue_df")


Rank Variance Per Country 

Compare the total number of comments made by users in each country during December 2019 and January 2020. For each month, rank countries by their total number of comments in descending order. Countries with the same total should share the same rank, and the next rank should increase by one (without skipping numbers). Return the names of the countries whose rank improved from December to January (that is, their rank number became smaller).

Tables:
- fb_comments_count -> how many comments each user made on a specific day

user_id	date	comments
18	2019-12-29	1
18	2020-01-02	1
58	2020-01-04	1


- fb_active_users -> which country each user belongs to

user_id	country
18	USA
21	Mali
32	Mali

goal:

- calculate total comments in dec 2019 and jan 2020
- rank countries in each month: highest comment will get rank 1
- find countries where january rank is less than december rank